In [1]:
!pip install sentence-transformers

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from lightgbm import LGBMRegressor
from sentence_transformers import SentenceTransformer

In [3]:
train_df = pd.read_csv("/content/train.csv")

train_df.shape

(75000, 4)

In [4]:
X = train_df["catalog_content"]
y = train_df["price"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_valid.shape)

(60000,)
(15000,)


In [5]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
X_train_embeddings = model.encode(
    X_train.tolist(),
    show_progress_bar=True
)

X_valid_embeddings = model.encode(
    X_valid.tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/1875 [00:00<?, ?it/s]

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

In [7]:
print(X_train_embeddings.shape)
print(X_valid_embeddings.shape)

(60000, 384)
(15000, 384)


In [8]:
lgbm_model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

lgbm_model.fit(
    X_train_embeddings,
    y_train
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.314623 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 97920
[LightGBM] [Info] Number of data points in the train set: 60000, number of used features: 384
[LightGBM] [Info] Start training from score 23.598634


LGBMRegressor(learning_rate=0.05, n_estimators=300, random_state=42)

In [9]:
predictions = lgbm_model.predict(
    X_valid_embeddings
)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [10]:
mae = mean_absolute_error(
    y_valid,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_valid,
        predictions
    )
)

r2 = r2_score(
    y_valid,
    predictions
)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 15.88905193337205
RMSE: 34.942537565198585
R2 Score: 0.19097725778223484


In [11]:
import re

def extract_quantity_features(text):
    text = str(text).lower()

    ounce = re.search(r'(\d+\.?\d*)\s*(oz|ounce)', text)
    pound = re.search(r'(\d+\.?\d*)\s*(lb|pound)', text)
    pack = re.search(r'pack of (\d+)', text)
    serving = re.search(r'(\d+)\s*servings', text)
    count = re.search(r'(\d+)\s*count', text)

    return pd.Series([
        float(ounce.group(1)) if ounce else 0,
        float(pound.group(1)) if pound else 0,
        float(pack.group(1)) if pack else 0,
        float(serving.group(1)) if serving else 0,
        float(count.group(1)) if count else 0
    ])


quantity_features = train_df[
    "catalog_content"
].apply(extract_quantity_features)

quantity_features.columns = [
    "ounce_feature",
    "pound_feature",
    "pack_feature",
    "serving_feature",
    "count_feature"
]

quantity_features.head()

,ounce_feature,pound_feature,pack_feature,serving_feature,count_feature
0,12.00,0.0,6.0,0.0,0.0
1,8.00,0.0,4.0,0.0,0.0
2,1.90,0.0,6.0,0.0,0.0
3,11.25,0.0,0.0,0.0,0.0
4,12.70,0.0,0.0,0.0,0.0


In [12]:
from scipy.sparse import hstack
import numpy as np

quantity_train = quantity_features.loc[
    X_train.index
].values

quantity_valid = quantity_features.loc[
    X_valid.index
].values

X_train_final = np.hstack([
    X_train_embeddings,
    quantity_train
])

X_valid_final = np.hstack([
    X_valid_embeddings,
    quantity_valid
])

print(X_train_final.shape)
print(X_valid_final.shape)

(60000, 389)
(15000, 389)


In [13]:
lgbm_hybrid = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

lgbm_hybrid.fit(
    X_train_final,
    y_train
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.324583 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 98612
[LightGBM] [Info] Number of data points in the train set: 60000, number of used features: 389
[LightGBM] [Info] Start training from score 23.598634


LGBMRegressor(learning_rate=0.05, n_estimators=300, random_state=42)

In [14]:
hybrid_predictions = lgbm_hybrid.predict(
    X_valid_final
)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [15]:
mae = mean_absolute_error(
    y_valid,
    hybrid_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_valid,
        hybrid_predictions
    )
)

r2 = r2_score(
    y_valid,
    hybrid_predictions
)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 14.666566864890877
RMSE: 33.23346396652572
R2 Score: 0.2681820290485003
